# LangChain Multi-Query for RAG

<div style="border:1px solid #ccc; border-radius:6px; padding:12px;">

<br>
<b>About</b><br><br>

This notebook is derived from the following notebook, with modifications and extensions: https://github.com/AI-Engineering-bootcamp/ai-eng-nbs-public/blob/master/langchain-multi-query-202503.ipynb
</div>

## Introduction
One of the advanced features in LangChain is Multi-Query for RAG, which allows for more efficient and effective information retrieval from large datasets.

## What is Multi-Query for RAG?

Multi-Query for RAG is a technique used to enhance the retrieval process in retrieval-augmented generation systems. It involves generating multiple queries from a single input and using these queries to retrieve a more diverse and relevant set of documents from the database. This approach improves the quality of the retrieved information, leading to better and more accurate generated responses.

## Key Components

1. **Query Generation**: Instead of generating a single query, the system creates multiple queries based on different aspects or perspectives of the input. These queries can be variations or expansions of the original input.

2. **Document Retrieval**: Each generated query is used to search the indexed documents. The results from all queries are combined to form a more comprehensive set of retrieved documents.

3. **Response Generation**: The retrieved documents are then used to generate the final response. By incorporating information from multiple queries, the generated response is more likely to be accurate and relevant.

## Benefits of Multi-Query for RAG

- **Improved Diversity**: Multiple queries increase the chances of retrieving diverse pieces of information, covering different aspects of the input topic.
- **Enhanced Relevance**: Combining results from multiple queries can lead to a more relevant and contextually appropriate set of documents.
- **Robustness**: Multi-query retrieval is more robust to variations and ambiguities in the input, as different queries can capture different interpretations.

## Example Workflow

1. **Input Processing**: The user provides an input query.
2. **Query Generation**: The system generates multiple queries from the input.
3. **Document Retrieval**: Each query is used to search the document index, and the results are combined.
4. **Response Generation**: The retrieved documents are used by the language model to generate the final response.

<div>
<img src="https://education-team-2020.s3.eu-west-1.amazonaws.com/ai-eng/images-langchain-4/muti.avif" alt='auto' width="1000"/>
</div>

## Example Code

Here is a simplified example of how Multi-Query for RAG might be implemented using LangChain:

```python
from langchain.chains import MultiQueryRAGChain
from langchain.prompts import PromptTemplate
from langchain.llms import OpenAI
from langchain.indexes import VectorIndex

# Define the base prompt template
base_prompt_template = PromptTemplate(
    input_variables=["input_text"],
    template="Generate multiple queries from the following input: {input_text}"
)

# Initialize the language model
llm = OpenAI(model="text-davinci-003")

# Create the Multi-Query RAG Chain
multi_query_chain = MultiQueryRAGChain(
    prompt=base_prompt_template,
    llm=llm,
    index=VectorIndex()  # Assuming VectorIndex is already populated with documents
)

# Input text
input_text = "Tell me about the history of artificial intelligence."

# Run the chain and get the response
response = multi_query_chain.run({"input_text": input_text})
print(response)


## Getting Data

We will download an existing dataset from Hugging Face Datasets.

In [1]:
!pip install "datasets>=4.4,<5" "pandas<3" "numpy<2" "pyarrow<24" "langchain-community<0.3"

  Using cached datasets-4.8.5-py3-none-any.whl.metadata (19 kB)
  Using cached pandas-2.3.3-cp311-cp311-macosx_11_0_arm64.whl.metadata (91 kB)
  Using cached pyarrow-23.0.1-cp311-cp311-macosx_12_0_arm64.whl.metadata (3.1 kB)
  Using cached filelock-4.2.4-py3-none-any.whl.metadata (2.0 kB)
  Using cached dill-0.4.1-py3-none-any.whl.metadata (10 kB)
  Using cached xxhash-4.0.2-cp311-cp311-macosx_11_0_arm64.whl.metadata (17 kB)
  Using cached multiprocess-0.70.19-py311-none-any.whl.metadata (7.5 kB)
  Using cached huggingface_hub-1.33.0-py3-none-any.whl.metadata (16 kB)
  Using cached click-8.5.0-py3-none-any.whl.metadata (2.6 kB)
  Using cached hf_xet-1.7.1-cp38-abi3-macosx_11_0_arm64.whl.metadata (4.9 kB)
Using cached datasets-4.8.5-py3-none-any.whl (528 kB)
Using cached pandas-2.3.3-cp311-cp311-macosx_11_0_arm64.whl (10.8 MB)
Using cached pyarrow-23.0.1-cp311-cp311-macosx_12_0_arm64.whl (34.3 MB)
Using cached dill-0.4.1-py3-none-any.whl (120 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
from datasets import load_dataset

data = load_dataset("jamescalam/ai-arxiv-chunked", split="train")
data

/Users/luis/Desktop/ironhack_sept26/demos_ai_engineering_sept2026/.venv/langchain-v0.2/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Dataset({
    features: ['doi', 'chunk-id', 'chunk', 'id', 'title', 'summary', 'source', 'authors', 'categories', 'comment', 'journal_ref', 'primary_category', 'published', 'updated', 'references'],
    num_rows: 41584
})

In [3]:
from langchain.docstore.document import Document

docs = []

for row in data:
    doc = Document(
        page_content=row["chunk"],
        metadata={
            "title": row["title"],
            "source": row["source"],
            "id": row["id"],
            "chunk-id": row["chunk-id"],
            "text": row["chunk"]
        }
    )
    docs.append(doc)

In [4]:
from dotenv import load_dotenv, find_dotenv
import os
_ = load_dotenv(find_dotenv())

OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY')
HUGGINGFACEHUB_API_TOKEN = os.getenv('HUGGINGFACEHUB_API_TOKEN')

# initialize connection to pinecone (get API key at app.pinecone.io)
api_key = os.getenv("PINECONE_API_KEY") or "YOUR_API_KEY"

## Embedding and Vector DB Setup

Initialize our embedding model:

In [5]:
import os
from getpass import getpass
from langchain.embeddings.openai import OpenAIEmbeddings

model_name = "text-embedding-ada-002"

embed = OpenAIEmbeddings(
    model=model_name, openai_api_key=OPENAI_API_KEY, disallowed_special=()
)

/var/folders/x1/00w2xr_j197gk698c1ljm3300000gn/T/ipykernel_72610/2914920317.py:7: LangChainDeprecationWarning: The class `OpenAIEmbeddings` was deprecated in LangChain 0.0.9 and will be removed in 1.0. An updated version of the class exists in the langchain-openai package and should be used instead. To use it run `pip install -U langchain-openai` and import as `from langchain_openai import OpenAIEmbeddings`.
  embed = OpenAIEmbeddings(


Now we create our vector DB to store our vectors. For this we need to get a [free Pinecone API key](https://app.pinecone.io) — the API key can be found in the "API Keys" button found in the left navbar of the Pinecone dashboard.

In [6]:
# !pip uninstall -y -q pinecone

!pip install -qU "langchain-pinecone<0.2" "pinecone-notebooks==0.1.1"


[notice] A new release of pip is available: 24.0 -> 26.2.2
[notice] To update, run: pip install --upgrade pip


In [7]:
from pinecone import Pinecone
# configure client
pc = Pinecone(api_key=api_key)

Now we setup our index specification, this allows us to define the cloud provider and region where we want to deploy our index. You can find a list of all [available providers and regions here](https://docs.pinecone.io/docs/projects).

In [8]:
from pinecone import ServerlessSpec

spec = ServerlessSpec(
    cloud="aws", region="us-east-1"
)

Creating an index, we set `dimension` equal to to dimensionality of Ada-002 (`1536`), and use a `metric` also compatible with Ada-002 (this can be either `cosine` or `dotproduct`). We also pass our `spec` to index initialization.

In [9]:
import time

index_name = "langchain-multi-query-demo"
existing_indexes = [
    index_info["name"] for index_info in pc.list_indexes()
]

# check if index already exists (it shouldn't if this is first time)
if index_name not in existing_indexes:
    # if does not exist, create index
    pc.create_index(
        index_name,
        dimension=1536,  # dimensionality of ada 002
        metric='dotproduct',
        spec=spec
    )
    # wait for index to be initialized
    while not pc.describe_index(index_name).status['ready']:
        time.sleep(1)

# connect to index
index = pc.Index(index_name)
time.sleep(1)
# view index stats
index.describe_index_stats()

{'dimension': 1536,
 'index_fullness': 0.0,
 'namespaces': {},
 'total_vector_count': 0}

Populate our index:

In [10]:
len(docs)

41584

In [11]:
# if you want to speed things up to follow along
#docs = docs[:5000]

In [12]:
!pip install tiktoken


[notice] A new release of pip is available: 24.0 -> 26.2.2
[notice] To update, run: pip install --upgrade pip


In [13]:
from tqdm.auto import tqdm

batch_size = 100

for i in tqdm(range(0, len(docs), batch_size)):
    i_end = min(len(docs), i+batch_size)
    docs_batch = docs[i:i_end]
    # get IDs
    ids = [f"{doc.metadata['id']}-{doc.metadata['chunk-id']}" for doc in docs_batch]
    # get text and embed
    texts = [d.page_content for d in docs_batch]
    embeds = embed.embed_documents(texts=texts)
    # get metadata
    metadata = [d.metadata for d in docs_batch]
    to_upsert = zip(ids, embeds, metadata)
    index.upsert(vectors=to_upsert)

100%|██████████| 416/416 [10:27<00:00,  1.51s/it]


## Multi-Query with LangChain

Now we switch across to using our populated index as a vectorstore in Langchain.

In [19]:
from langchain.vectorstores import Pinecone

text_field = "text"

vectorstore = Pinecone(index, embed.embed_query, text_field)

/Users/luis/Desktop/ironhack_sept26/demos_ai_engineering_sept2026/.venv/langchain-v0.2/lib/python3.11/site-packages/langchain_community/vectorstores/pinecone.py:68: UserWarning: Passing in `embedding` as a Callable is deprecated. Please pass in an Embeddings object instead.
  warnings.warn(


In [24]:
from langchain.chat_models import ChatOpenAI

llm = ChatOpenAI(
    openai_api_key=OPENAI_API_KEY,
    model='gpt-6-luna',
    temperature=1,  # gpt-6-luna only accepts the default temperature
)

We initialize the `MultiQueryRetriever`:

In [25]:
from langchain.retrievers.multi_query import MultiQueryRetriever

retriever = MultiQueryRetriever.from_llm(
    retriever=vectorstore.as_retriever(), llm=llm
)

We set logging so that we can see the queries as they're generated by our LLM.

In [26]:
# Set logging for the queries
import logging

logging.basicConfig()
logging.getLogger("langchain.retrievers.multi_query").setLevel(logging.INFO)

To query with our multi-query retriever we call the `get_relevant_documents` method.

In [27]:
question = "tell me about llama 2?"

docs = retriever.get_relevant_documents(query=question)
len(docs)

INFO:langchain.retrievers.multi_query:Generated queries: ['What is Meta’s Llama 2 language model, and what are its key features and capabilities?', 'How was Llama 2 trained, and how does it compare with other large language models?', 'What are Llama 2’s licensing terms, intended uses, and limitations?']


6

From this we get a variety of docs retrieved by each of our queries independently. By default the `retriever` is returning `3` docs for each query — totalling `9` documents — however, as there is some overlap we actually return `6` unique docs.

In [28]:
docs

[Document(metadata={'chunk-id': '1', 'id': '2307.09288', 'source': 'http://arxiv.org/pdf/2307.09288', 'title': 'Llama 2: Open Foundation and Fine-Tuned Chat Models'}, page_content='Alan Schelten Ruan Silva Eric Michael Smith Ranjan Subramanian Xiaoqing Ellen Tan Binh Tang\nRoss Taylor Adina Williams Jian Xiang Kuan Puxin Xu Zheng Yan Iliyan Zarov Yuchen Zhang\nAngela Fan Melanie Kambadur Sharan Narang Aurelien Rodriguez Robert Stojnic\nSergey Edunov Thomas Scialom\x03\nGenAI, Meta\nAbstract\nIn this work, we develop and release Llama 2, a collection of pretrained and ﬁne-tuned\nlarge language models (LLMs) ranging in scale from 7 billion to 70 billion parameters.\nOur ﬁne-tuned LLMs, called L/l.sc/a.sc/m.sc/a.sc /two.taboldstyle-C/h.sc/a.sc/t.sc , are optimized for dialogue use cases. Our\nmodels outperform open-source chat models on most benchmarks we tested, and based on\nourhumanevaluationsforhelpfulnessandsafety,maybeasuitablesubstituteforclosedsource models. We provide a detailed 

## Adding the Generation in RAG

So far we've built a multi-query powered **R**etrieval **A**ugmentation chain. Now, we need to add **G**eneration.

In [29]:
from langchain.prompts import PromptTemplate
from langchain.chains import LLMChain

QA_PROMPT = PromptTemplate(
    input_variables=["query", "contexts"],
    template="""You are a helpful assistant who answers user queries using the
    contexts provided. If the question cannot be answered using the information
    provided say "I don't know".

    Contexts:
    {contexts}

    Question: {query}""",
)

# Chain
qa_chain = LLMChain(llm=llm, prompt=QA_PROMPT)

/var/folders/x1/00w2xr_j197gk698c1ljm3300000gn/T/ipykernel_72610/541099612.py:17: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 1.0. Use RunnableSequence, e.g., `prompt | llm` instead.
  qa_chain = LLMChain(llm=llm, prompt=QA_PROMPT)


In [30]:
out = qa_chain(
    inputs={
        "query": question,
        "contexts": "\n---\n".join([d.page_content for d in docs])
    }
)
out["text"]

/var/folders/x1/00w2xr_j197gk698c1ljm3300000gn/T/ipykernel_72610/2730685832.py:1: LangChainDeprecationWarning: The method `Chain.__call__` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use invoke instead.
  out = qa_chain(


'Llama 2 is Meta’s family of large language models, released in pretrained and fine-tuned versions ranging from 7 billion to 70 billion parameters. The fine-tuned models, called **Llama 2-Chat**, are designed for dialogue and assistant-style use; the pretrained models can be adapted for other text-generation tasks.\n\nMeta reported that Llama 2-Chat performed better than many open-source chat models on the benchmarks it tested and was comparable to some closed-source models in its human evaluations. It is intended for English-language commercial and research use, under a custom license and an Acceptable Use Policy. Meta also released safety guidance and invited community feedback to help improve the models.'

## Chaining Everything with a SequentialChain

We can pull together the logic above into a function or set of methods, whatever is prefered — however if we'd like to use LangChain's approach to this we must "chain" together multiple chains. The first retrieval component is (1) not a chain per se, and (2) requires processing of the output. To do that, and fit with LangChain's "chaining chains" approach, we setup the _retrieval_ component within a `TransformChain`:

In [31]:
from langchain.chains import TransformChain

def retrieval_transform(inputs: dict) -> dict:
    docs = retriever.get_relevant_documents(query=inputs["question"])
    docs = [d.page_content for d in docs]
    docs_dict = {
        "query": inputs["question"],
        "contexts": "\n---\n".join(docs)
    }
    return docs_dict

retrieval_chain = TransformChain(
    input_variables=["question"],
    output_variables=["query", "contexts"],
    transform=retrieval_transform
)

Now we chain this with our generation step using the `SequentialChain`:

In [32]:
from langchain.chains import SequentialChain

rag_chain = SequentialChain(
    chains=[retrieval_chain, qa_chain],
    input_variables=["question"],  # we need to name differently to output "query"
    output_variables=["query", "contexts", "text"]
)

Then we perform the full RAG pipeline:

In [33]:
out = rag_chain({"question": question})
out["text"]

INFO:langchain.retrievers.multi_query:Generated queries: ['What is Llama 2, and what are its main features and capabilities?', 'Can you explain Meta’s Llama 2 language model, including its model sizes and intended uses?', 'How does Llama 2 work, and how does it compare with other large language models?']


'Llama 2 is a family of large language models released by Meta, with versions ranging from 7 billion to 70 billion parameters. It includes **pretrained models** for adapting to different text-generation tasks and **Llama 2-Chat** models fine-tuned for assistant-style dialogue.\n\nAccording to the provided context, Llama 2-Chat performed better than existing open-source chat models on most benchmarks Meta tested, and appeared comparable to some closed-source models in the human evaluations reported. The models were intended for commercial and research use in English, subject to Meta’s license and acceptable-use requirements.'

---

## Custom Multiquery

We'll try this with two prompts, both encourage more variety in search queries.

**Prompt A**
```
Your task is to generate 3 different search queries that aim to
answer the user question from multiple perspectives.
Each query MUST tackle the question from a different viewpoint,
we want to get a variety of RELEVANT search results.
Provide these alternative questions separated by newlines.
Original question: {question}
```


**Prompt B**
```
Your task is to generate 3 different search queries that aim to
answer the user question from multiple perspectives. The user questions
are focused on Large Language Models, Machine Learning, and related
disciplines.
Each query MUST tackle the question from a different viewpoint, we
want to get a variety of RELEVANT search results.
Provide these alternative questions separated by newlines.
Original question: {question}
```

In [34]:
from typing import List
from langchain.chains import LLMChain
from pydantic import BaseModel, Field
from langchain.prompts import PromptTemplate
from langchain.output_parsers import PydanticOutputParser


# Output parser will split the LLM result into a list of queries
class LineList(BaseModel):
    # "lines" is the key (attribute name) of the parsed output
    lines: List[str] = Field(description="Lines of text")


class LineListOutputParser(PydanticOutputParser):
    def __init__(self) -> None:
        super().__init__(pydantic_object=LineList)

    def parse(self, text: str) -> LineList:
        lines = text.strip().split("\n")
        return LineList(lines=lines)


output_parser = LineListOutputParser()

template = """
Your task is to generate 3 different search queries that aim to
answer the user question from multiple perspectives. The user questions
are focused on Large Language Models, Machine Learning, and related
disciplines.
Each query MUST tackle the question from a different viewpoint, we
want to get a variety of RELEVANT search results.
Provide these alternative questions separated by newlines.
Original question: {question}
"""

QUERY_PROMPT = PromptTemplate(
    input_variables=["question"],
    template=template,
)
llm = ChatOpenAI(temperature=0, openai_api_key=OPENAI_API_KEY)

# Chain
llm_chain = LLMChain(llm=llm, prompt=QUERY_PROMPT, output_parser=output_parser)

In [37]:
from typing import List

from langchain_core.output_parsers import BaseOutputParser
from langchain_core.prompts import PromptTemplate
from pydantic import BaseModel, Field


# Output parser will split the LLM result into a list of queries
class LineListOutputParser(BaseOutputParser[List[str]]):
    """Output parser for a list of lines."""

    def parse(self, text: str) -> List[str]:
        lines = text.strip().split("\n")
        return list(filter(None, lines))  # Remove empty lines


output_parser = LineListOutputParser()

QUERY_PROMPT = PromptTemplate(
    input_variables=["question"],
    template = """
    Your task is to generate 3 different search queries that aim to
    answer the user question from multiple perspectives. The user questions
    are focused on Large Language Models, Machine Learning, and related
    disciplines.
    Each query MUST tackle the question from a different viewpoint, we
    want to get a variety of RELEVANT search results.
    Provide these alternative questions separated by newlines.
    Original question: {question}
    """,
)

llm = ChatOpenAI(
    openai_api_key=OPENAI_API_KEY,
    model='gpt-6-luna',
    temperature=1,  # gpt-6-luna only accepts the default temperature
)

# Chain
llm_chain = QUERY_PROMPT | llm | output_parser

# Other inputs
question = "What are the key features and capabilities of Large Language Model Llama 2?"

In [38]:
# Run
retriever = MultiQueryRetriever(
    retriever=vectorstore.as_retriever(), llm_chain=llm_chain, parser_key="lines"
)  # "lines" is the key (attribute name) of the parsed output

# Results
unique_docs = retriever.invoke("How does Llama 2 compare to other Large Language Models in terms of performance and efficiency?")
len(unique_docs)

INFO:langchain.retrievers.multi_query:Generated queries: ['Llama 2 vs GPT-4, Claude, PaLM 2, and other open LLMs: benchmark performance on reasoning, coding, knowledge, and instruction-following tasks', 'Llama 2 efficiency compared with other large language models: training compute, inference latency, memory requirements, and performance per GPU', 'How does Llama 2 compare with Mistral, Falcon, and other open-weight models across model sizes, benchmarks, hardware requirements, and deployment costs?']


11

In [39]:
unique_docs

[Document(metadata={'chunk-id': '26', 'id': '2304.03277', 'source': 'http://arxiv.org/pdf/2304.03277', 'title': 'Instruction Tuning with GPT-4'}, page_content='models have drawn a signiﬁcant interest and promoted work towards open, general-purpose, textbased assistants that are aligned with human values. Early attempts on foundation LLMs include\nBLOOM (Scao et al., 2022), GPT-J (Wang & Komatsuzaki, 2021), GPT-NEO (Black et al., 2021)\nOPT (Zhang et al., 2022) and LLaMA (Zhang et al., 2023). To align LLMs with chat-based\nassistance, Open-Assistant (LAION-AI, 2023) is built on GPT-J, and Alpaca/Vicuna are built on\nLLaMA. Furthermore, OpenFlamingo (Awadalla et al., 2023) and LLaMA-Adapter (Zhang et al.,\n2023) connect LLaMA with image inputs, paving a way to build open-source multi-modal LLMs.\n6 C ONCLUSIONS\nThis paper demonstrates the effectiveness of instruction tuning using GPT-4. We release 52K English\nand Chinese instruction-following instances generated using GPT-4 as well as 

Putting this together in another `SequentialChain`:

In [40]:
retrieval_chain = TransformChain(
    input_variables=["question"],
    output_variables=["query", "contexts"],
    transform=retrieval_transform
)

rag_chain = SequentialChain(
    chains=[retrieval_chain, qa_chain],
    input_variables=["question"],  # we need to name differently to output "query"
    output_variables=["query", "contexts", "text"]
)

And asking again:

In [41]:
out = rag_chain({"question": question})
out["text"]

INFO:langchain.retrievers.multi_query:Generated queries: ['Llama 2 model sizes, architecture, context length, training data, and key capabilities', 'Llama 2 Chat instruction tuning, dialogue performance, and safety features compared with other LLMs', 'Llama 2 benchmark results, practical use cases, hardware requirements, and commercial licensing terms']


'Llama 2 is a family of large language models from Meta, with **7B to 70B parameters**. Its key features include:\n\n- **Two model types:** pretrained models that can be adapted for different text-generation tasks, and fine-tuned **Llama 2-Chat** models optimized for assistant-style dialogue.\n- **Dialogue and instruction use:** The chat models are designed to respond in conversational settings.\n- **Benchmark performance:** The paper reports that Llama 2-Chat generally outperformed the open-source chat models tested. In the authors’ human evaluations, it was also on par with some closed-source models for helpfulness and safety.\n- **Safety work:** The authors describe their fine-tuning and safety approach and say they plan to improve safety with community feedback.\n- **Intended use:** Commercial and research use in English, subject to the applicable license and acceptable-use requirements. The stated intended uses do not include languages other than English.'

After finishing, delete your Pinecone index to save resources:

In [42]:
pc.delete_index(index_name)

---